In [1]:
from pathlib import Path
import pandas as pd
from PyDI.io import load_parquet
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_colwidth', 100)

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
amazon_sample = load_parquet(
    DATA_DIR / "amazon_sample.parquet",
    name="amazon_sample"
)

goodreads_sample = load_parquet(
    DATA_DIR / "goodreads_sample.parquet",
    name="goodreads_sample"
)

metabooks_sample = load_parquet(
  DATA_DIR / "metabooks_sample.parquet",
  name="metabooks_sample"
)

In [3]:
from PyDI.io import load_csv

train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [4]:
from PyDI.entitymatching import EmbeddingBlocker

embedding_blocker_m2a = EmbeddingBlocker(
    metabooks_sample, amazon_sample,
    text_cols=['title', 'author'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)


embedding_blocker_m2g = EmbeddingBlocker(
    metabooks_sample, goodreads_sample,
    text_cols=['title', 'author'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

embedding_candidates_m2a = embedding_blocker_m2a.materialize()
embedding_candidates_m2g = embedding_blocker_m2g.materialize()

In [5]:
from PyDI.entitymatching import EntityMatchingEvaluator

# evaluate blocking: metabooks -> amazon
results_m2a = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=embedding_candidates_m2a,
    blocker=embedding_blocker_m2a,
    test_pairs=train_m2a,
    out_dir=BLOCK_EVAL_DIR
)
# evaluate blocking: metabooks -> goodreads
results_m2g = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=embedding_candidates_m2g,
    blocker=embedding_blocker_m2g,
    test_pairs=train_m2g,
    out_dir=BLOCK_EVAL_DIR
)

display(results_m2a)
display(results_m2g)



{'pair_completeness': 0.9945652173913043,
 'pair_quality': 0.00026165321940694794,
 'reduction_ratio': 0.9994290620408163,
 'total_candidates': 699399,
 'total_possible_pairs': 1225000000,
 'true_positives_found': 183,
 'total_true_pairs': 184,
 'evaluation_timestamp': '2025-11-20T14:55:05.326993',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

{'pair_completeness': 0.9875,
 'pair_quality': 0.0002265703981472851,
 'reduction_ratio': 0.9994307306122449,
 'total_candidates': 697355,
 'total_possible_pairs': 1225000000,
 'true_positives_found': 158,
 'total_true_pairs': 160,
 'evaluation_timestamp': '2025-11-20T14:55:37.917413',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

In [9]:
from PyDI.entitymatching import RuleBasedMatcher
from PyDI.entitymatching import StringComparator, NumericComparator

comparators_m2a = [
    StringComparator(column='title',similarity_function='cosine',tokenization="ngram_3"),
    StringComparator(column='author',similarity_function='jaro_winkler', preprocess=str.lower,tokenization="char"),
    StringComparator(column='publisher',similarity_function='jaro_winkler', preprocess=str.lower,tokenization="char"),
    NumericComparator(column='publish_year'),
]
comparators_m2g=[StringComparator(column='title',similarity_function='cosine',tokenization='ngram_3'),
    StringComparator(column='author',similarity_function='cosine', preprocess=str.lower,tokenization="word"),
    StringComparator(column='publisher',similarity_function='jaro_winkler', preprocess=str.lower,tokenization="char"),
    NumericComparator(column='publish_year'),
]

matcher = RuleBasedMatcher()

# matching metabooks > amazon
correspondences_m2a = matcher.match(
    df_left=metabooks_sample,
    df_right=amazon_sample, 
    candidates=embedding_blocker_m2a,
    comparators=comparators_m2a,
    weights=[0.45, 0.25, 0.25,0.05], 
    threshold=0.65,
    id_column='id'
)

# matching metabooks > goodreads
correspondences_m2g = matcher.match(
    df_left=metabooks_sample,
    df_right=goodreads_sample, 
    candidates=embedding_blocker_m2g,
    comparators=comparators_m2g,

    weights=[0.45, 0.25,0.25,0.05], 
    threshold=0.65,
    id_column='id'
)

In [10]:
debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

# evaluate matching metabooks -> amazon
eval_results_m2a = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2a,
    test_pairs=test_m2a,
    out_dir=debug_output_dir
)

display(eval_results_m2a)

{'precision': 0.6818181818181818,
 'recall': 0.9782608695652174,
 'f1': 0.8035714285714285,
 'accuracy': 0.89,
 'true_positives': 45,
 'false_positives': 21,
 'false_negatives': 1,
 'true_negatives': 133,
 'threshold_used': 0.0,
 'total_correspondences': 39186,
 'filtered_correspondences': 39186,
 'evaluation_timestamp': '2025-11-20T15:14:59.089580',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [11]:
# evaluate matching metabooks -> goodreads
eval_results_m2g = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2g,
    test_pairs=test_m2g,
    out_dir=debug_output_dir
)

display(eval_results_m2g)

{'precision': 0.7659574468085106,
 'recall': 0.9,
 'f1': 0.8275862068965516,
 'accuracy': 0.925,
 'true_positives': 36,
 'false_positives': 11,
 'false_negatives': 4,
 'true_negatives': 149,
 'threshold_used': 0.0,
 'total_correspondences': 8452,
 'filtered_correspondences': 8452,
 'evaluation_timestamp': '2025-11-20T15:14:59.946903',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [12]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)
cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,19321,86.746285
1,3,825,3.704036
2,4,1381,6.200332
3,5,177,0.794684
4,6,237,1.064069
...,...,...,...
45,102,2,0.008979
46,134,1,0.004490
47,284,1,0.004490
48,396,1,0.004490



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5539,86.398378
1,3,597,9.312120
2,4,117,1.824988
3,5,47,0.733115
4,6,36,0.561535
5,7,24,0.374357
6,8,11,0.171580
7,9,3,0.046795
8,10,10,0.155982
9,11,4,0.062393


In [13]:
from PyDI.entitymatching import MaximumBipartiteMatching, StableMatching

# We are using Maxmimum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2a = clusterer.cluster(correspondences_m2a)
mbm_correspondences_m2g = clusterer.cluster(correspondences_m2g)


cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)

cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2a,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2g,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,6700,100.0



📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,26561,100.0


In [14]:
all_correspondences = pd.concat([mbm_correspondences_m2a, mbm_correspondences_m2g], ignore_index=True)

len(all_correspondences)

33261

In [15]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

metabooks_sample.attrs["trust_score"] = 3
goodreads_sample.attrs["trust_score"] = 2
amazon_sample.attrs["trust_score"] = 1
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('author', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('language', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('numratings', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres',union)

In [16]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_sn_blocker.jsonl")

fused_rb_emblocker = engine.run(
    datasets=[amazon_sample, metabooks_sample, goodreads_sample],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False
)

print(f'Fused rows: {len(fused_rb_emblocker):,}')


Fused rows: 30,381


In [17]:
fused_rb_emblocker.to_parquet(PIPELINE_DIR / "fused_rb_emblocker.parquet")

In [18]:
from PyDI.fusion import tokenized_match, boolean_match,numeric_tolerance_match,set_equality_match

import numpy as np
import re, ast, numpy as np, pandas as pd


def categories_set_equal(a, b) -> bool:
    """Return True if a and b contain the same unique categories (order/type agnostic)."""
    def to_set(x):
        def items(v):
            # missing
            if v is None or (isinstance(v, float) and np.isnan(v)): return []
            # numpy array → recurse over elements
            if isinstance(v, np.ndarray): 
                out=[]; [out.extend(items(e)) for e in v.flatten()]; return out
            # python containers → recurse over elements
            if isinstance(v, (list, tuple, set)):
                out=[]; [out.extend(items(e)) for e in v]; return out
            # scalar/string: try parse stringified list; else split by delimiters
            s = str(v).strip()
            if s == "" or s.lower() in {"nan","none"}: return []
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)): return items(parsed)
            except Exception:
                pass
            return [p.strip() for p in re.split(r"[|,;/]", s) if p.strip()]
        return {it.lower() for it in items(x)}
    return to_set(a) == to_set(b)

fused_dataset = load_parquet(PIPELINE_DIR / "fused_rb_emblocker.parquet")
fused_dataset["publish_year"] = fused_dataset["publish_year"].astype("Int16")
fused_dataset["page_count"] = fused_dataset["page_count"].astype("Int32")
golden_fused_dataset= load_parquet(MLDS_DIR / "golden_fused_books.parquet")

strategy.add_evaluation_function("title", tokenized_match)
strategy.add_evaluation_function("author", tokenized_match)
strategy.add_evaluation_function("publisher", tokenized_match)
strategy.add_evaluation_function("publish_year", numeric_tolerance_match)
strategy.add_evaluation_function("numratings", numeric_tolerance_match)
strategy.add_evaluation_function("price", numeric_tolerance_match)
strategy.add_evaluation_function("page_count", numeric_tolerance_match)
strategy.add_evaluation_function("language", tokenized_match)
strategy.add_evaluation_function("genres", categories_set_equal)

In [19]:
from PyDI.fusion import DataFusionEvaluator
fused_dataset.drop_duplicates(subset='isbn_clean', keep='first',inplace=True)
# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused_dataset,
    fused_id_column='isbn_clean',
    gold_df=golden_fused_dataset,
    gold_id_column='isbn_clean',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.733
  macro_accuracy: 0.733
  num_evaluated_records: 38828
  num_evaluated_attributes: 9
  total_evaluations: 342014
  total_correct: 250579
  genres_accuracy: 0.709
  genres_count: 37448
  price_accuracy: 0.728
  price_count: 36440
  author_accuracy: 0.709
  author_count: 38828
  title_accuracy: 0.741
  title_count: 38828
  publisher_accuracy: 0.744
  publisher_count: 38828
  language_accuracy: 0.760
  language_count: 38695
  numratings_accuracy: 0.742
  numratings_count: 38828
  publish_year_accuracy: 0.729
  publish_year_count: 38708
  page_count_accuracy: 0.730
  page_count_count: 35411

Overall Accuracy: 73.3%
